# Serve extraction and SDLC explanation agents from Colab

Run all cells top to bottom. The notebook loads the already-saved extraction LoRA adapter from Google Drive and serves both agents from one runtime. Copy the printed public ngrok URL into `HF_COLAB_ENDPOINT_URL` in `backend/.env`.

This notebook does not train a model. The SDLC endpoint disables the extraction adapter and only explains the Python-computed ranking; it cannot return or replace scores.


In [ ]:
%pip -q install -U "transformers>=4.51" "peft>=0.15" "bitsandbytes>=0.45" "accelerate>=1.6" "fastapi>=0.115" "uvicorn>=0.34" "pyngrok>=7.2" "httpx>=0.28" "huggingface_hub>=0.30"


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
from huggingface_hub import login
from pyngrok import ngrok

drive.mount('/content/drive')
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
if not NGROK_AUTH_TOKEN:
    raise RuntimeError('Add NGROK_AUTH_TOKEN in Colab Secrets, then rerun this cell.')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

MODEL_ID = 'Qwen/Qwen3-1.7B'
ADAPTER_DIR = Path('/content/drive/MyDrive/loan-origination/qwen3-1.7b-loan-requirements-lora-v2')
if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f'Saved extraction adapter not found at {ADAPTER_DIR}. Run the training notebook save cell first.')
print('Base model:', MODEL_ID)
print('Extraction adapter:', ADAPTER_DIR)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun this notebook.')
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN or None)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map='auto',
    dtype=compute_dtype,
    token=HF_TOKEN or None,
)
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), is_trainable=False)
model.eval()
print('Loaded model and extraction adapter on:', model.device)


In [ ]:
import json
from typing import Any
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title='Loan Origination Model Agents')

class ExtractRequest(BaseModel):
    system: str
    user: str
    max_tokens: int = 1500

class RankingEntry(BaseModel):
    method: str
    percentage: float

class SdlcRequest(BaseModel):
    requirements: list[dict[str, Any]]
    ranking: list[RankingEntry]
    project_summary: str = ''

def generate_text(system: str, user: str, max_tokens: int, *, disable_adapter: bool = False) -> str:
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': user},
    ]
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors='pt',
        return_dict=True,
    ).to(model.device)
    limit = max(1, min(int(max_tokens), 4096))
    with torch.inference_mode():
        if disable_adapter:
            with model.disable_adapter():
                generated = model.generate(
                    **encoded, max_new_tokens=limit, do_sample=False,
                    eos_token_id=tokenizer.eos_token_id,
                )
        else:
            generated = model.generate(
                **encoded, max_new_tokens=limit, do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
            )
    new_tokens = generated[0, encoded['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def first_json_object(text: str) -> dict:
    start = text.find('{')
    if start < 0:
        raise ValueError('No JSON object found in model output.')
    depth = 0
    in_string = False
    escaped = False
    for index in range(start, len(text)):
        char = text[index]
        if in_string:
            if escaped:
                escaped = False
            elif char == '\\':
                escaped = True
            elif char == '"':
                in_string = False
            continue
        if char == '"':
            in_string = True
        elif char == '{':
            depth += 1
        elif char == '}':
            depth -= 1
            if depth == 0:
                parsed = json.loads(text[start:index + 1])
                if not isinstance(parsed, dict):
                    raise ValueError('First JSON value is not an object.')
                return parsed
    raise ValueError('JSON object in model output was incomplete.')

@app.get('/health')
def health():
    return {'status': 'ok', 'endpoints': ['/extract', '/sdlc']}

@app.post('/extract')
def extract(request: ExtractRequest):
    try:
        text = generate_text(request.system, request.user, request.max_tokens)
        if not text:
            raise ValueError('Model returned empty output.')
        return {'text': text}
    except Exception as exc:
        raise HTTPException(status_code=422, detail=f'Extraction generation failed: {exc}') from exc

@app.post('/sdlc')
def explain_sdlc(request: SdlcRequest):
    system = (
        'You explain a ranking of software development methods whose percentages were computed by a separate deterministic Python scorer. '
        'Explain why each method received its supplied percentage using only the provided requirements and project summary. '
        'Discuss trade-offs and risks for each method. Do not calculate, revise, reorder, or replace percentages. '
        'Do not choose, recommend, or declare one method the winner. Do not invent project facts. '
        'Return only one JSON object with exactly these fields: summary (string), '
        'per_method (array of objects with method and explanation strings), risks (array of strings).'
    )
    ranking = [entry.model_dump() for entry in request.ranking]
    user = (
        'Explain this deterministic ranking. Keep every method name as supplied. '
        'The percentage values are authoritative and must remain unchanged; the response schema has no score fields.\n'
        + 'Project summary: ' + (request.project_summary or 'Not supplied') + '\n'
        + 'Requirements (JSON): ' + json.dumps(request.requirements, ensure_ascii=False) + '\n'
        + 'Ranking (JSON): ' + json.dumps(ranking, ensure_ascii=False)
    )
    try:
        raw = generate_text(system, user, 1200, disable_adapter=True)
        result = first_json_object(raw)
        if not isinstance(result.get('summary'), str) or not result['summary'].strip():
            raise ValueError('summary must be a non-empty string')
        per_method = result.get('per_method')
        expected = [entry['method'] for entry in ranking]
        if not isinstance(per_method, list) or any(
            not isinstance(item, dict)
            or not isinstance(item.get('method'), str)
            or not isinstance(item.get('explanation'), str)
            for item in per_method
        ):
            raise ValueError('per_method must contain method and explanation strings')
        if [item['method'] for item in per_method] != expected:
            raise ValueError('per_method methods must match the supplied ranking in order')
        risks = result.get('risks')
        if not isinstance(risks, list) or any(not isinstance(item, str) for item in risks):
            raise ValueError('risks must be a list of strings')
        return {'summary': result['summary'], 'per_method': per_method, 'risks': risks}
    except Exception as exc:
        raise HTTPException(status_code=422, detail=f'SDLC response could not be parsed or validated: {exc}') from exc


In [ ]:
import threading
import time
import uvicorn

server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning'))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
for _ in range(60):
    if server.started:
        break
    time.sleep(1)
if not server.started:
    raise RuntimeError('FastAPI server did not start on port 8000.')
tunnel = ngrok.connect(8000, 'http')
PUBLIC_URL = tunnel.public_url


In [ ]:
import httpx

test_description = 'A consumer loan app shall let applicants submit an application and let loan officers review it.'
extract_test = httpx.post(
    PUBLIC_URL + '/extract',
    json={
        'system': 'Extract concise atomic requirements. Return JSON only.',
        'user': 'Client description:\n' + test_description,
        'max_tokens': 400,
    },
    timeout=180,
)
sdlc_test = httpx.post(
    PUBLIC_URL + '/sdlc',
    json={
        'requirements': [{'statement': 'The system shall let applicants submit a loan application.'}],
        'ranking': [{'method': 'Agile', 'percentage': 72.5}, {'method': 'Spiral', 'percentage': 68.0}],
        'project_summary': test_description,
    },
    timeout=180,
)
print('GET /health:', httpx.get(PUBLIC_URL + '/health', timeout=30).status_code)
print('POST /extract:', extract_test.status_code, list(extract_test.json().keys()))
print('POST /sdlc:', sdlc_test.status_code, list(sdlc_test.json().keys()))


In [ ]:
print('Public ngrok URL:', PUBLIC_URL)
print('Copy this URL into HF_COLAB_ENDPOINT_URL in backend/.env (no trailing path).')
